# Import

In [70]:
import torch
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoTokenizer
import transformers
import numpy as np

In [58]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.dataset import NLIDataset
from src.model import get_model

In [59]:
MAX_LENGTH = 128
MODEL_NAME = "vinai/phobert-base"
BATCH_SIZE = 32
transformers.logging.set_verbosity_error()

# Inference

In [60]:
device = torch.device("cpu")
device

device(type='cpu')

In [61]:
splits = {'train': 'vianli_train.jsonl', 'validation': 'vianli_dev.jsonl', 'test': 'vianli_test.jsonl'}
df_test = pd.read_json("hf://datasets/uitnlp/ViANLI/" + splits["test"], lines=True)
df_test.head(5)

,uid,premise,hypothesis,label
0,uit_Adver_2078_2_21_11,"Sáng 23/5, ông Trần Văn Vịnh, Chủ tịch UBND ph...",Theo thông tin một công chức Nhà nước ở phường...,contradiction
1,uit_Adver_1669_2_21_09,Cảnh sát thành phố Bhopal hôm 13/5 cho biết sự...,Sự việc gây sốc xảy ra đầu tháng 4 và nghi phạ...,contradiction
2,uit_Adver_712_2_11_04,"Ngày 2/5, sinh nhật tuổi 49 của The Rock, tài ...",Cung hoàng đạo của The Rock là Kim Ngưu.,entailment
3,uit_Adver_1663_3_31_09,"Sau khi xem xét camera giám sát trong khu vực,...",Cảnh sát đã phán đoán bằng nhận định sắc bén s...,neutral
4,uit_Adver_1561_4_21_08,"3h sáng cùng ngày, trái tim của của người hiến...","3am, một người được tái sinh.",contradiction


In [62]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, num_labels=len(df_test["label"].unique()))

test_dataset = NLIDataset(test_df, tokenizer, max_length=MAX_LENGTH)

test_dataloader = DataLoader(
    test_dataset,
    batch_size = BATCH_SIZE,
    shuffle = False
)
len(test_dataloader)

32

In [63]:
model = get_model(MODEL_NAME, num_labels=len(df_test["label"].unique()))
model.load_state_dict(torch.load("../model/best_model.pth", map_location=device))
model.eval()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [64]:
print("Start inferencing...")
all_probs = []
all_preds = []
with torch.inference_mode(): 
    for batch in tqdm(test_dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids = input_ids, attention_mask = attention_mask)
        logits = outputs.logits

        probs = torch.softmax(logits, dim=1)
        max_probs, preds = torch.max(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(max_probs.cpu().numpy())

Start inferencing...


100%|██████████| 32/32 [01:12<00:00,  2.28s/it]


In [65]:
all_preds[:5], all_probs[:5]

([np.int64(0), np.int64(2), np.int64(0), np.int64(0), np.int64(0)],
 [np.float32(0.78468204),
  np.float32(0.5306065),
  np.float32(0.6350706),
  np.float32(0.41491327),
  np.float32(0.83628404)])

In [ ]:
df_test["predicted_labels"] = all_preds
df_test["confidence_score"] = all_probs

reverse_map = {v: k for k, v in test_dataset.label_dict.items()}

df_test["predicted_labels_text"] = df_test["predicted_labels"].map(reverse_map)
df_test.head(10)

,uid,premise,hypothesis,label,predicted_labels,confidence_score,predicted_labels_text
0,uit_Adver_2078_2_21_11,"Sáng 23/5, ông Trần Văn Vịnh, Chủ tịch UBND ph...",Theo thông tin một công chức Nhà nước ở phường...,contradiction,0,0.784682,contradiction
1,uit_Adver_1669_2_21_09,Cảnh sát thành phố Bhopal hôm 13/5 cho biết sự...,Sự việc gây sốc xảy ra đầu tháng 4 và nghi phạ...,contradiction,2,0.530607,neutral
2,uit_Adver_712_2_11_04,"Ngày 2/5, sinh nhật tuổi 49 của The Rock, tài ...",Cung hoàng đạo của The Rock là Kim Ngưu.,entailment,0,0.635071,contradiction
3,uit_Adver_1663_3_31_09,"Sau khi xem xét camera giám sát trong khu vực,...",Cảnh sát đã phán đoán bằng nhận định sắc bén s...,neutral,0,0.414913,contradiction
4,uit_Adver_1561_4_21_08,"3h sáng cùng ngày, trái tim của của người hiến...","3am, một người được tái sinh.",contradiction,0,0.836284,contradiction
5,uit_Adver_596_1_31_03,Sau khi trường Cao đẳng Sư phạm Ninh Thuận sáp...,Trường Cao đẳng Sư phạm Ninh Thuận sẽ mở thêm ...,neutral,1,0.612923,entailment
6,uit_Adver_320_4_31_02,"Ông Phạm Cao Vỹ, chủ tịch Hiệp hội Du lịch Sa ...",Ông Phạm Cao Vỹ được yêu thích.,neutral,0,0.729441,contradiction
7,uit_Adver_1595_1_31_08,Ngành y tế Quảng Nam ngày 16/5 ghi nhận 3.070 ...,Giữa tháng 5 số lượng mẫu âm tính tại Quảng Na...,neutral,1,0.598760,entailment
8,uit_Adver_1419_3_11_08,"Các bệnh nhân 3262, 3268, 3660 tiếp tục cách l...",Các bệnh nhận được quan sát điều trị trong vòn...,entailment,0,0.783121,contradiction
9,uit_Adver_1479_5_11_08,"""Bệnh viện cũng chuẩn bị 20 hòm phiếu được dán...",20 thùng rỗng được đảm bảo cho cuộc bỏ phiếu.,entailment,0,0.612285,contradiction
